# DAY 12- Streaming_Deduplication

- **Streaming deduplication** — How to prevent duplicate transactions from being processed.
- **Checkpoint recovery** — How Spark resumes after a job stops or fails.
- **Exactly-once processing** — What guarantees Spark provides and where duplicates can still occur.

## Steps- 
1. **Create** DataFrame
2. **Read** the Data stream
3. **Apply Streaming DEDUPLICATE** logic 

````
        dfDedup = (
                    dfStream
                    .dropDuplicates(["transactionId"])
                )
````

4. **Write** the dataStream to Silver table
5. **Verify** the data.


**What is stored in a checkpoint?**

Typically, a Structured Streaming checkpoint contains information such as:
- Source offsets or progress information.
- Commit information for completed micro-batches.
- State-store data for stateful operations such as deduplication and aggregation.
- Metadata needed to recover the streaming query.

The checkpoint must be stored in durable storage accessible when the job restarts.

**What if we delete the checkpoint?**

- If you delete the checkpoint and restart the stream, Spark generally **treats it as a new streaming query.**
- For a Delta source, this can cause previously processed data to be read again, depending on the source configuration.
- If the destination already contains those records,** duplicates may be introduced.**

**Never delete a production checkpoint simply to fix a streaming error without understanding the recovery implications.**

## Exactly-once vs at-least-once processing


**Guarantee	Meaning**
```
At-most-once	A record is processed no more than once, but some records may be lost
At-least-once	Records are retried if needed, so duplicates are possible
Exactly-once	Each input record has one effective result in the destination, under the supported source/sink guarantees
````

Spark Structured Streaming can provide end-to-end exactly-once fault tolerance with replayable sources, checkpoints, and compatible sinks such as Delta.

However, **exactly-once processing does not mean business-level deduplication.**

For example, if the source contains two separate records with transaction ID 102, both may be processed exactly once unless you explicitly apply deduplication.

Similarly, **foreachBatch provides at-least-once write semantics by default.** External writes must be made idempotent to avoid duplicate effects after retries.

## Q1. How do you handle duplicate records in Spark Structured Streaming?
- In Structured Streaming, I can use **_dropDuplicates on a business key_** such as transaction ID. 
- Spark maintains **state across micro-batches** to recognize previously seen keys. 
- For long-running streams, I would consider **_watermark-based deduplication_** to control state growth, depending on the business requirements and the supported runtime.

## Q2. What is the purpose of checkpointing?
- Checkpointing stores **the progress and state** of a streaming query. 
- If the job fails or restarts, Spark can **_resume from the checkpoint_** instead of reprocessing everything from the beginning. 
- It is especially important for stateful operations like aggregations and deduplication.

## Q3. Does exactly-once processing guarantee no duplicate transactions?
- **No**. 
- Exactly-once processing means each input record has one effective processing result under the supported guarantees. 
- If the source itself contains duplicate business transactions, those are still separate input records. 
- We need business-key deduplication to remove them.

### 1. A streaming DataFrame uses dropDuplicates(["transactionId"]) without a watermark. What is a potential concern?

A. Spark cannot process any data \
B. Deduplication state can grow indefinitely  **--> Correct** \
C. The source Delta table is overwritten \
D. Duplicates are never removed 

Correct - Without watermark-based state eviction, Spark may retain previously seen transaction keys indefinitely.


### 2. A streaming query fails after successfully committing several micro-batches. What helps it resume?

A. ORDER BY \
B. The checkpoint  **--> Correct** \
C. DROP TABLE \
D. VACUUM 

Correct-Checkpoints preserve source progress and, where needed, state for recovery.


### 3. A Delta source contains two separate records with the same transactionId. Does exactly-once processing automatically remove one?

A. Yes, always \
B. Only in Complete mode \
C. No, business-key deduplication is separate  **--> Correct** \
D. Only if the table is partitioned 

Correct-Exactly-once fault tolerance and business-record deduplication are different guarantees.